<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.2-cloud-run-deploy/practice/GCP_Capstone_7.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 7.2 — Deploy to Cloud Run with IAM

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Authenticate with Application Default Credentials, set the active project + region, and enable the Cloud Run / build / secrets APIs. Run this once before any exercise below.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE to your project id
REGION = 'us-central1'

!gcloud config set project $PROJECT_ID
!gcloud config set run/region $REGION
print(f'Project: {PROJECT_ID}')

In [ ]:
%%bash
# Enable the APIs Cloud Run source-deploys depend on
gcloud services enable \
  run.googleapis.com \
  artifactregistry.googleapis.com \
  cloudbuild.googleapis.com \
  secretmanager.googleapis.com

# Source deploys build as the Compute Engine default service account; on new projects it
# lacks source-bucket / Artifact Registry access, so `gcloud run deploy --source` 403s.
# Grant it the Cloud Build builder role (idempotent; allow ~1-2 min to propagate).
PROJECT_NUMBER=$(gcloud projects describe "$(gcloud config get-value project)" --format='value(projectNumber)')
gcloud projects add-iam-policy-binding "$(gcloud config get-value project)" \
  --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
  --role="roles/cloudbuild.builds.builder" --condition=None

## Exercise 1: Create Project Files

**Difficulty:** Easy

Create pyproject.toml + Dockerfile + server.py.

1. pyproject.toml with fastmcp
2. Dockerfile with python:3.13-slim + uv
3. server.py from 7.1

In [ ]:
import os
os.makedirs('documind-mcp', exist_ok=True)

with open('documind-mcp/pyproject.toml', 'w') as f:
    f.write('[project]\nname = "documind-mcp-server"\nversion = "0.1.0"\nrequires-python = ">=3.10"\ndependencies = ["fastmcp>=4,<5"]\n')

with open('documind-mcp/Dockerfile', 'w') as f:
    f.write('FROM python:3.13-slim\nCOPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/\nCOPY . /app\nWORKDIR /app\nENV PYTHONUNBUFFERED=1\nRUN uv sync\nEXPOSE $PORT\nCMD ["uv", "run", "server.py"]\n')

print(f'Files: {os.listdir("documind-mcp")}')

In [ ]:
# server.py — the 4-tool DocuMind MCP server from 7.1, listening on $PORT for Cloud Run
server = '''import asyncio, os, logging
from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from typing import Literal
logging.basicConfig(level=logging.INFO)
mcp = FastMCP("DocuMind")
DOCS = [{"id":"doc-001","title":"Q4 Financial Report","cat":"financial","pages":24},
        {"id":"doc-002","title":"Engineering Design","cat":"technical","pages":18},
        {"id":"doc-003","title":"Employee Handbook","cat":"hr","pages":45},
        {"id":"doc-004","title":"Marketing Strategy","cat":"marketing","pages":12}]
@mcp.tool
def search_documents(query: str, max_results: int = 5) -> list[dict]:
    """Search documents by keyword."""
    if not query.strip(): raise ToolError("Query empty")
    return [{"id":d["id"],"title":d["title"]} for d in DOCS if query.lower() in d["title"].lower()][:max_results]
@mcp.tool
def calculate_cost(page_count: int, processing_type: Literal["standard","premium","enterprise"]="standard") -> dict:
    """Calculate processing cost."""
    if page_count<=0: raise ToolError("Positive pages required")
    rates={"standard":0.01,"premium":0.03,"enterprise":0.05}
    return {"cost":round(rates[processing_type]*page_count,2)}
@mcp.tool
def get_stats() -> dict:
    """Get repository stats."""
    return {"total":len(DOCS),"pages":sum(d["pages"] for d in DOCS)}
@mcp.tool
def classify_document(title: str, content: str) -> dict:
    """Classify a document."""
    if not title.strip(): raise ToolError("Title required")
    kws={"financial":["revenue","budget"],"technical":["api","system"],"hr":["employee"],"marketing":["campaign"]}
    text=(title+" "+content).lower()
    scores={c:sum(1 for k in ws if k in text) for c,ws in kws.items()}
    return {"category":max(scores,key=scores.get)}
if __name__=="__main__":
    port=int(os.getenv("PORT",8080))
    asyncio.run(mcp.run_async(transport="streamable-http",host="0.0.0.0",port=port))
'''
with open('documind-mcp/server.py','w') as f: f.write(server)
print('server.py created')

## Exercise 2: Deploy to Cloud Run

**Difficulty:** Easy

gcloud run deploy --source . --no-allow-unauthenticated.

1. Enable APIs
2. Deploy
3. Get URL

In [ ]:
%%bash
# APIs were enabled in Setup. Source-deploy the folder (Cloud Build builds the image), private by default.
cd documind-mcp
gcloud run deploy documind-mcp-server \
  --no-allow-unauthenticated \
  --quiet \
  --region=us-central1 \
  --source .

# Print the service URL
gcloud run services describe documind-mcp-server \
  --region=us-central1 \
  --format='value(status.url)'

## Exercise 3: Test with Proxy

**Difficulty:** Easy

gcloud run services proxy. Inspector to localhost:8080/mcp.

1. Start proxy
2. Connect Inspector
3. List 4 tools

In [ ]:
# Show the MCP endpoint + the proxy command. The proxy injects your identity token
# so a local MCP Inspector can reach the private service at http://localhost:8080/mcp
import subprocess
url = subprocess.check_output(
    'gcloud run services describe documind-mcp-server --region=us-central1 --format="value(status.url)"',
    shell=True).decode().strip()
print(f'MCP endpoint: {url}/mcp')
print('Run locally, then point MCP Inspector at http://localhost:8080/mcp:')
print('  gcloud run services proxy documind-mcp-server --region=us-central1 --port=8080')

In [ ]:
%%bash
# The proxy is long-running (blocks the cell). Run it in a local terminal, or launch it
# in the background here, then connect MCP Inspector to http://localhost:8080/mcp and
# confirm the 4 tools (search_documents, calculate_cost, get_stats, classify_document).
gcloud run services proxy documind-mcp-server --region=us-central1 --port=8080 &
sleep 8
curl -s http://localhost:8080/mcp \
  -H 'Content-Type: application/json' \
  -H 'Accept: application/json, text/event-stream' \
  -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}'

## Exercise 4: curl with Auth

**Difficulty:** Medium

curl with Authorization: Bearer identity-token.

1. Get token
2. curl with header
3. Parse response

In [ ]:
%%bash
# Hit the private service directly (no proxy) using a Google-signed identity token.
TOKEN=$(gcloud auth print-identity-token)
SERVICE_URL=$(gcloud run services describe documind-mcp-server --region=us-central1 --format='value(status.url)')

curl -s "$SERVICE_URL/mcp" \
  -H "Authorization: Bearer $TOKEN" \
  -H 'Content-Type: application/json' \
  -H 'Accept: application/json, text/event-stream' \
  -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}'

# Without the Authorization header the same request returns HTTP 403 (proves IAM is enforced).

## Exercise 5: Scale-to-Zero

**Difficulty:** Medium

Deploy min-instances=0. Wait. Observe cold start.

1. Deploy
2. Wait 15 min
3. Check logs

In [ ]:
%%bash
# min-instances=0 means you pay $0 while idle; the first request after idle pays a cold start.
gcloud run services update documind-mcp-server \
  --region=us-central1 \
  --min-instances=0 \
  --max-instances=3

gcloud run services describe documind-mcp-server \
  --region=us-central1 \
  --format='value(spec.template.metadata.annotations)'

In [ ]:
%%bash
# After the service has been idle ~15 min, send one request, then read logs for the
# 'startupProbe'/container-start lines — the cold start is typically ~1-2s for this image.
gcloud run services logs read documind-mcp-server \
  --region=us-central1 \
  --limit=30

## Exercise 6: Secret Manager

**Difficulty:** Medium

Create secret. Deploy --set-secrets. Read via os.getenv.

1. gcloud secrets create
2. Deploy with --set-secrets
3. Verify access

In [ ]:
%%bash
# 1. Create the secret and add a version
printf 'super-secret-value-123' | gcloud secrets create documind-api-key \
  --data-file=- --replication-policy=automatic

# 2. Let the Cloud Run runtime service account read it
PROJECT_NUMBER=$(gcloud projects describe "$(gcloud config get-value project)" --format='value(projectNumber)')
gcloud secrets add-iam-policy-binding documind-api-key \
  --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
  --role='roles/secretmanager.secretAccessor'

# 3. Mount the latest version as env var DOCUMIND_API_KEY (server reads it via os.getenv)
cd documind-mcp
gcloud run deploy documind-mcp-server \
  --no-allow-unauthenticated \
  --quiet \
  --region=us-central1 \
  --source . \
  --set-secrets=DOCUMIND_API_KEY=documind-api-key:latest

## Exercise 7: MCP Toolbox

**Difficulty:** Challenge

Write tools.yaml. Deploy Toolbox container.

1. Define source + tools
2. Store in Secret Manager
3. Deploy

In [ ]:
# tools.yaml: one Cloud SQL Postgres source + one SQL-backed tool
# PREREQUISITE (OPTIONAL - Toolbox path only): create the Cloud SQL Postgres source this Toolbox
# connects to. These are SHELL commands (run in a %%bash cell or with a ! prefix, NOT as Python).
# A Cloud SQL instance is BILLABLE (~a few $/day) and takes ~5-10 min; delete it when done
# (!gcloud sql instances delete documind-instance --quiet). The cell below only WRITES tools.yaml.
#   !gcloud sql instances create documind-instance --database-version=POSTGRES_15 --region=us-central1 --cpu=1 --memory=4GiB
#   !gcloud sql databases create documents --instance=documind-instance   (then load a `documents` table)
toolbox = f'''kind: source
name: documind_db
type: cloud-sql-postgres
project: {PROJECT_ID}
region: us-central1
instance: documind-instance
database: documents
---
kind: tool
name: search-docs-db
type: postgres-sql
source: documind_db
description: Search documents by keyword.
parameters:
  - name: query
    type: string
statement: SELECT id,title FROM documents WHERE title ILIKE $1 LIMIT 10;
'''
with open('tools.yaml','w') as f: f.write(toolbox)
print('tools.yaml created')

In [ ]:
%%bash
# Store the config in Secret Manager, then run Google's prebuilt Toolbox image on Cloud Run,
# mounting the secret and pointing --tools-file at it.
gcloud secrets create documind-tools --data-file=tools.yaml --replication-policy=automatic

PROJECT_NUMBER=$(gcloud projects describe "$(gcloud config get-value project)" --format='value(projectNumber)')
gcloud secrets add-iam-policy-binding documind-tools \
  --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
  --role='roles/secretmanager.secretAccessor'

gcloud run deploy documind-toolbox \
  --image=us-central1-docker.pkg.dev/database-toolbox/toolbox/toolbox:latest \
  --region=us-central1 \
  --no-allow-unauthenticated \
  --quiet \
  --set-secrets=/app/tools.yaml=documind-tools:latest \
  --args='--tools-file,/app/tools.yaml,--address,0.0.0.0,--port,8080'

## Exercise 8: Production Deploy

**Difficulty:** Challenge

Full: IAM + secrets + cpu-boost + service account.

1. All flags
2. Service account
3. Verify

In [ ]:
%%bash
# 1. Dedicated least-privilege runtime service account
gcloud iam service-accounts create documind-run \
  --display-name='DocuMind Cloud Run runtime' || true
PROJECT_ID=$(gcloud config get-value project)
RUN_SA="documind-run@${PROJECT_ID}.iam.gserviceaccount.com"

# 2. Grant it only the secret it needs
gcloud secrets add-iam-policy-binding documind-api-key \
  --member="serviceAccount:${RUN_SA}" \
  --role='roles/secretmanager.secretAccessor'

# 3. Full production deploy: private, custom SA, secret mount, startup CPU boost, bounded scaling
cd documind-mcp
gcloud run deploy documind-mcp-server \
  --no-allow-unauthenticated \
  --quiet \
  --region=us-central1 \
  --source . \
  --service-account="${RUN_SA}" \
  --set-secrets=DOCUMIND_API_KEY=documind-api-key:latest \
  --cpu-boost \
  --min-instances=0 \
  --max-instances=5 \
  --memory=512Mi \
  --concurrency=40

# Verify the running config
gcloud run services describe documind-mcp-server --region=us-central1 \
  --format='value(spec.template.spec.serviceAccountName)'